# Homework 07: Implement LSTM (group 4)

## 2.1.0 Importing the dataset

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import datetime
import tensorflow_datasets as tfds

# load the data
(train_ds, test_ds), ds_info = tfds.load('mnist', split =['train', 'test'], as_supervised = True, with_info = True)

# only use a subset of the data to accelerate the training process
train_ds = train_ds.take(5000)
test_ds = test_ds.take(500)

c:\Users\marta\anaconda3\envs\iannwtf\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2.1.1 Prepare the dataset

In [24]:
def new_target_fnc(ds, sequence_len):
    """
    Creates list of new targets by alternately adding and subtracting
    The first digit is added, the second subtracted, the third added, etc
    Parameters
    ----------
    ds : TensorFlowDataset
    original mnist dataset containg images and targets as a tuple.
    sequence_len : int
    indicates at which point the sum has to reset for the new sequence
    Returns
    -------
    l : list
    list containing the new targets
    """
    l = list()
    for i, elem in enumerate(ds):
        if (i % sequence_len) == 0:
            l.append(int(elem[1]))
        else:
            if (i % 2) == 0:
                l.append(int(l[i-1] + elem[1]))
            else:
                l.append(int(l[i-1] - elem[1]))
    return l

# Generate the new targets from the old ones for sequence length 28
train_new_targets = new_target_fnc(train_ds, sequence_len=28)

# Convert the list of new targets to a dataset
train_new_targets_ds = tf.data.Dataset.from_tensor_slices(train_new_targets)

# Zip the original dataset and the new targets dataset
zipped_ds = tf.data.Dataset.zip((train_ds, train_new_targets_ds))

## 2.1.2 Creating data pipeline

In [26]:
import tensorflow as tf

@tf.function
def prepare_data(mnist, batch_size, sequence_len):
  """
  This function is used to prepare the raw data for training and testing them.

  Arguments:
  mnist -- (a subset of) the MNIST dataset
  batch_size -- denotes the batch size 
  seq_len -- length of the time sequence

  Returns a modified data set that includes tuples containing two images and the respective target.
  """
  # Use the map() method to drop the old targets and keep only the new ones
  mnist = zipped_ds.map(lambda img, target: (img[0], target))
  # flatten the images into vectors and replace old targets
  mnist = mnist.map(lambda img, target: (tf.reshape(img, (-1,)), target))
  # convert images to float32 data type
  mnist = mnist.map(lambda img, target: (tf.cast(img, tf.float32), target))
  # project the pixel values into range [-1, 1]
  mnist = mnist.map(lambda img, target: ((img/128.)-1., target))
  # cache progress into memory
  mnist = mnist.cache()
  # use .batch() to create time sequences
  mnist = mnist.batch(sequence_len)
  # shuffle the data into a random order
  mnist = mnist.shuffle(1000)
  # use batches of size 32
  mnist = mnist.batch(batch_size)
  # prefetch as many data points as we put into a batch
  mnist = mnist.prefetch(tf.data.AUTOTUNE)
  
  return mnist

prepare_data(train_ds, batch_size=32, sequence_len=28)

<_VariantDataset element_spec=(TensorSpec(shape=(None, None, 784), dtype=tf.float32, name=None), TensorSpec(shape=(None, None), dtype=tf.int32, name=None))>